In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os

try:
    path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
    data = pd.read_csv(os.path.join(path, 'Monday-WorkingHours.pcap_ISCX.csv'))
    print(" Path to dataset files:", path)
    
except Exception as e:
    print(f"Something failed... {e}")

data.head()

 Path to dataset files: /Users/antoniogonzalez/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data.columns = data.columns.str.strip()

data['Label'] = le.fit_transform(data['Label'])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

X = data.drop('Label', axis=1)
y = data['Label']

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Drop redundant/duplicate columns from X before training
# Note: must drop from X here, not from data, since X was already created above
columns_drop = [
    'Fwd Header Length.1',   # duplicate column (renamed by pandas on load)
    'Avg Fwd Segment Size',
    'Subflow Fwd Bytes',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Subflow Bwd Bytes',
    'Packet Length Variance'
]
X = X.drop(columns=columns_drop, errors='ignore')

# Split FIRST before the scaler ever sees the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# Fit scaler ONLY on training data, then apply to test separately
# This prevents test set statistics from leaking into training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#  FloatTensor = decimal numbers (features)
#  LongTensor = whole numbers (labels/classes)
X_train_t = torch.FloatTensor(X_train_scaled)  # practice exam
X_test_t  = torch.FloatTensor(X_test_scaled)   # actual exam
y_train_t = torch.LongTensor(y_train.values)   # practice answers
y_test_t  = torch.LongTensor(y_test.values)    # actual answers

class NeuralNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NeuralNet, self).__init__()
        self.layers = nn.Sequential(
            #  Layer 1: project features up to 128 neurons
            nn.Linear(input_size, 128),
            nn.ReLU(),  # sets negative values to zero, passes positive ones unchanged
            nn.Linear(128, 64),  # Layer 2: compress 128 neurons down to 64
            nn.ReLU(),
            nn.Linear(64, num_classes)  # output layer: one score per class
        )
    def forward(self, x):
        return self.layers(x)

#  input_size = number of feature columns after drop
input_size = X_train_t.shape[1]

#  num_classes = number of unique attack labels in the dataset
num_classes = len(y.unique())

model = NeuralNet(input_size, num_classes)

#  CrossEntropyLoss measures how wrong the model's predictions were
#  (lower loss = better predictions)
criterion = nn.CrossEntropyLoss()

# Adam optimizer adjusts model weights after each batch
# lr = 0.001 controls how large each adjustment step is
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#  Wrap training data into a dataset object for PyTorch
dataset = TensorDataset(X_train_t, y_train_t)

#  DataLoader feeds data in batches of 512 rows
#  shuffle=True re-orders data each epoch to prevent overfitting
loader = DataLoader(dataset, batch_size=512, shuffle=True)

for epoch in range(10):
    model.train()
    for X_batch, y_batch in loader:
        optimizer.zero_grad()              # reset gradients from previous batch
        output = model(X_batch)            # forward pass: make predictions
        loss = criterion(output, y_batch)  # calculate how wrong the model was
        loss.backward()                    # backward pass: compute gradients
        optimizer.step()                   # update weights
    print(f'Epoch {epoch+1}/10 Loss: {loss.item():.4f}')

    model.eval()
    with torch.no_grad():  # no gradient tracking during evaluation
        y_pred = model(X_test_t).argmax(dim=1).numpy()

    print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import shap

model.to('cpu')
X_train_cpu = X_train_t[:100].to("cpu")
X_test_cpu  = X_test_t[:100].to('cpu')
explainer = shap.DeepExplainer(model, X_train_cpu)
shap_values = explainer.shap_values(X_test_cpu)
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=X.columns, class_names=le.classes_, show=False)
plt.tight_layout()
plt.show()